# 20 — The milestone-managed book: does a model beat a careful analyst's lookup two years in?

Notebook 14 showed that, two years after the request, the landmark model ranks the requests still in the queue far better than the entry model and the plain lookup (criterion C8). This notebook asks the question a portfolio manager would ask next, as a registered test: if I keep 25 or 50 of the utility-scale requests still in the queue at that point, does the model's choice realise more operations and MW than the choice of a lookup that knows the same dated agreement status — and can the model's level be repaired at the landmark with outcomes already known that day?

**Inputs → Outputs**

| | Path | Notes |
|---|---|---|
| in | `data/interim/trackA.parquet`, `data/interim/queue_clean.parquet`, `results/dt_tuning.json` | notebooks 02/04/06 |
| in | `data/interim/milestones.parquet` | notebook 13 — dated agreements |
| in | `data/processed/predictions_v2.parquet` | notebook 11 — the fair baseline's entry P(operation by 5 years) |
| in | `data/processed/predictions_landmark.parquet` | notebook 14 — to check the refit reproduces the landmark model |
| out | `results/success_criteria_v4.csv`, `results/success_criteria_v4_registered.json` | criteria C15–C17, written before any test number (decision #58) |
| out | `results/managed_book_pool.csv` | the pool at the +2 landmark |
| out | `results/managed_books.csv`, `results/managed_book_bootstrap.csv` | each chooser's books, point and resampled |
| out | `results/managed_book_composition.csv` | what each book holds (agreement known, region) |
| out | `results/landmark_level_repair.csv`, `results/landmark_level_calibration.csv` | the level repair at the landmark (C17) |
| out | `results/success_criteria_v4_scored.csv` | the ten registered lines, scored |
| out | `figures/F18_managed_book.*`, `figures/20_managed_book/*` | F18 and the notebook's figures |
| out | `results/managed_book_checks.csv` | this notebook's checks |

## Setup

In [1]:
import hashlib
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))  # notebooks run from notebooks/; src/ is one level up
from src import config, discrete_time, drift, evaluation, landmark, paths, plots, portfolio, variants

paths.ensure_dirs()
plots.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK = "20_managed_book"
START = time.time()


def save(fig, name, caption):
    """Save a figure of this notebook under figures/20_managed_book/ with its caption stub."""
    return plots.save_fig(fig, name, caption, NOTEBOOK, folder=NOTEBOOK)

## 1. Registration (decision #58)

This cell is the registration. It was run and committed on its own before any later cell existed, so every rule below — the pool, the landmark, the book sizes, the cap, the choosers, the outcome, the resampling and the thesis rule — was fixed before a book was chosen or a number scored. On every later run the cell checks that the file on disk is the registered one and stops if anything differs.

Why these rules. The pool is notebook 14's +2 risk set restricted to 20 MW and more, the cohort of C6–C12. The landmark model's view is the one notebook 14 already published as best two years in ("agreement updated": the dated agreement flag, congestion frozen at entry because updating it is the known artefact). The decisive comparator is a lookup that **also knows whether the agreement has been executed**: beating the plain lookup would only repeat C8, since the model wins there on information a careful analyst also has. The count-limited book is the frame notebook 16's registered backtest lacked: ranking is all that decides it.

In [2]:
REGISTRATION = {
    "decided": "2026-10-08",
    "decision": 58,
    "n_boot": 200,
    "level": 0.95,
    "seed": 2058,
    "pool": "Track A test entrants 2017–2019 of 20 MW and more (variants.at_least_20) in the queue two full years after the request (landmark.risk_set(test, 2))",
    "landmark_years": 2,
    "horizon_after_landmark": 3,
    "book_sizes": [25, 50],
    "region_cap_share": 0.5,
    "tie_break": "a uniform random number per request, drawn once per resample and shared by every chooser; lower first",
    "outcome": "operation within 3 years of the landmark (event == 1 and duration − 2 ≤ 3), all observed by 2024-12-31; MW of those requests",
    "choosers": {
        "M": "landmark model: notebook 14's 'agreement updated' view (LightGBM hazards with the dated agreement flag, congestion at entry), refitted on entrants to 2016",
        "LA": "agreement-aware lookup: region × technology × size-band Aalen–Johansen cells from the landmark on, fitted separately on training requests with and without an agreement known at the landmark (landmark.agreement_lookup_at_landmark)",
        "E": "no re-scoring: the fair baseline's P(operation by 5 years) at entry (notebook 11)",
    },
    "reported_not_registered": ["L0: the plain lookup at the landmark (C8 already compares it)", "random: a random book under the same cap"],
    "level_repair": "for entry cohort c, one intercept (slope fixed at 1, drift.fit_intercept) fitted on cohort c's operations within 1 year and cohort c − 1's within 2 years, against the landmark model's entry view (landmark k = 0) of the same requests, all sizes and regions pooled; applied to M's +2 predictions of cohort c. Every outcome used ends before 1 January of c + 2, before any landmark of cohort c",
    "thesis_rule": "the stronger thesis (a model-managed book beats a careful analyst's lookup at the milestone) is supported only if both C15 lines against LA pass; otherwise the report keeps the current thesis",
}
lines = []
for metric, cid in [("operations", "C15"), ("MW", "C16")]:
    for bench in ["LA", "E"]:
        for n in REGISTRATION["book_sizes"]:
            lines.append({"id": cid, "line": f"+2 landmark, ≥ 20 MW, book of {n}: {metric} realised within 3 years, M vs {bench}",
                          "metric": metric, "n": n, "model": "M", "benchmark": bench,
                          "rule": "paired 95% interval of M − benchmark entirely above 0"})
for cohort in ["all sizes", "≥ 20 MW"]:
    lines.append({"id": "C17", "line": f"+2 landmark, {cohort}: M after the level repair, calibration intercept at 3 years",
                  "metric": "intercept", "n": None, "model": "M repaired", "benchmark": "",
                  "rule": "95% interval covers 0"})
REGISTRATION["criteria"] = lines
criteria = pd.DataFrame(lines)

reg_path = paths.RESULTS / "success_criteria_v4_registered.json"
text = json.dumps(REGISTRATION, indent=2, ensure_ascii=False)
if reg_path.exists():
    assert reg_path.read_text(encoding="utf-8") == text, "the registration differs from the committed file — a registered rule was changed"
else:
    reg_path.write_text(text, encoding="utf-8")
criteria.to_csv(paths.RESULTS / "success_criteria_v4.csv", index=False)
REGISTERED_SHA = hashlib.sha256(reg_path.read_bytes()).hexdigest()
print(f"{len(criteria)} registered lines; registration file SHA-256 {REGISTERED_SHA[:16]}…")
criteria[["id", "line", "rule"]]

10 registered lines; registration file SHA-256 0ea97d5aac9f208c…


,id,line,rule
0,C15,"+2 landmark, ≥ 20 MW, book of 25: operations r...",paired 95% interval of M − benchmark entirely ...
1,C15,"+2 landmark, ≥ 20 MW, book of 50: operations r...",paired 95% interval of M − benchmark entirely ...
2,C15,"+2 landmark, ≥ 20 MW, book of 25: operations r...",paired 95% interval of M − benchmark entirely ...
3,C15,"+2 landmark, ≥ 20 MW, book of 50: operations r...",paired 95% interval of M − benchmark entirely ...
4,C16,"+2 landmark, ≥ 20 MW, book of 25: MW realised ...",paired 95% interval of M − benchmark entirely ...
5,C16,"+2 landmark, ≥ 20 MW, book of 50: MW realised ...",paired 95% interval of M − benchmark entirely ...
6,C16,"+2 landmark, ≥ 20 MW, book of 25: MW realised ...",paired 95% interval of M − benchmark entirely ...
7,C16,"+2 landmark, ≥ 20 MW, book of 50: MW realised ...",paired 95% interval of M − benchmark entirely ...
8,C17,"+2 landmark, all sizes: M after the level repa...",95% interval covers 0
9,C17,"+2 landmark, ≥ 20 MW: M after the level repair...",95% interval covers 0
